# Day 9 — Python Scripts: Tasks, Docker, and Working with Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/kestra-certified/notebooks/day-09-python-scripts.ipynb#scrollTo=f1a2b3c4)

**Course:** Kestra for Data Engineers  
**Badge:** Practice  

Python is the lingua franca of data engineering. Today you'll go deep on Kestra's Python plugin: runner options, dependency management, multi-file tasks, output variables, and chaining Python tasks into a data pipeline.

**By the end of this notebook you will:**
- Write Python tasks that use pandas to read public data and compute statistics
- Publish named outputs back to Kestra using `Kestra.outputs()`
- Choose between PROCESS and DOCKER runners — and know the trade-offs
- Pass helper modules into a task using `inputFiles`
- Chain two Python tasks where the second consumes outputs from the first

In [ ]:
%pip install -q pyyaml pandas requests

## 1. Python Task Anatomy

A Kestra Python task has three zones of content:

```yaml
id: my_python_task
type: io.kestra.plugin.scripts.python.Commands
beforeCommands:          # shell commands run before the script
  - pip install pandas
script: |                # the Python source to execute
  import pandas as pd
  ...
  Kestra.outputs({...})  # publish output variables
inputFiles:              # files injected into working directory
  helper.py: |           # inline content or internal storage URI
    def clean(s): ...
outputFiles:             # glob patterns to capture
  - '*.csv'
env:                     # environment variables
  API_KEY: '{{ secret("API_KEY") }}'
runner: DOCKER           # DOCKER or PROCESS
docker:
  image: python:3.11-slim
```

In [ ]:
import yaml

# Minimal Python task
minimal_task = {
    "id": "basic_stats",
    "type": "io.kestra.plugin.scripts.python.Commands",
    "beforeCommands": ["pip install pandas -q"],
    "script": (
        "import pandas as pd\n"
        "url = 'https://people.sc.fsu.edu/~jburkardt/data/csv/airtravel.csv'\n"
        "df = pd.read_csv(url)\n"
        "df.columns = [c.strip() for c in df.columns]\n"
        "print(f'Rows: {len(df)}, Columns: {len(df.columns)}')\n"
        "print(df.describe().to_string())\n"
    )
}

flow = {
    "id": "python-basic",
    "namespace": "tutorial.day09",
    "tasks": [minimal_task]
}

print(yaml.dump(flow, default_flow_style=False, sort_keys=False))

## 2. Kestra.outputs() — The Bridge to Downstream Tasks

`Kestra.outputs(dict)` serialises key-value pairs into Kestra's output variable system. The function is injected by the Kestra worker — no import needed.

**Supported value types:** str, int, float, bool, list, dict (must be JSON-serialisable)

**Reference downstream:** `{{ outputs.<taskId>.vars.<key> }}`

In [ ]:
import pandas as pd
import json

# Simulate a Kestra Python task with Kestra.outputs()
url = 'https://people.sc.fsu.edu/~jburkardt/data/csv/airtravel.csv'
df = pd.read_csv(url)
df.columns = [c.strip() for c in df.columns]
numeric_df = df.select_dtypes(include='number')

outputs_payload = {
    "row_count": int(len(df)),
    "col_count": int(len(df.columns)),
    "columns": list(df.columns),
    "total_sum": float(numeric_df.values.sum()),
    "max_value": float(numeric_df.values.max())
}

# In Kestra: Kestra.outputs(outputs_payload)
print("Kestra.outputs() payload:")
print(json.dumps(outputs_payload, indent=2))
print()
print("Downstream Pebble references:")
for k, v in outputs_payload.items():
    print(f"  {{{{ outputs.basic_stats.vars.{k} }}}}  →  {v}")

In [ ]:
# Flow that uses Kestra.outputs()
outputs_flow = {
    "id": "python-with-outputs",
    "namespace": "tutorial.day09",
    "tasks": [
        {
            "id": "analyse",
            "type": "io.kestra.plugin.scripts.python.Commands",
            "beforeCommands": ["pip install pandas -q"],
            "script": (
                "import pandas as pd\n"
                "url = 'https://people.sc.fsu.edu/~jburkardt/data/csv/airtravel.csv'\n"
                "df = pd.read_csv(url)\n"
                "df.columns = [c.strip() for c in df.columns]\n"
                "numeric_df = df.select_dtypes(include='number')\n"
                "Kestra.outputs({\n"
                "    'row_count': int(len(df)),\n"
                "    'col_count': int(len(df.columns)),\n"
                "    'max_value': float(numeric_df.values.max()),\n"
                "    'columns': list(df.columns)\n"
                "})\n"
            )
        },
        {
            "id": "report",
            "type": "io.kestra.plugin.core.log.Log",
            "message": (
                "Dataset: {{ outputs.analyse.vars.row_count }} rows × "
                "{{ outputs.analyse.vars.col_count }} cols | "
                "max={{ outputs.analyse.vars.max_value }}"
            )
        }
    ]
}

print(yaml.dump(outputs_flow, default_flow_style=False, sort_keys=False))

## 3. PROCESS vs DOCKER Runner

| Dimension | PROCESS | DOCKER |
|-----------|---------|--------|
| Isolation | None — runs on Kestra worker | Full — separate container |
| Speed | Fast (no pull/start overhead) | Slower (first run pulls image) |
| Dependencies | Shared with worker Python env | Fully specified per task |
| Security | Worker processes share env vars | Isolated, controlled env |
| Best for | Dev/lightweight tasks | Production, heavy deps, custom libs |

DOCKER runner cached image pulls on the same worker node. After the first run, startup is near-instant.

In [ ]:
# PROCESS runner (default)
process_task = {
    "id": "lightweight_task",
    "type": "io.kestra.plugin.scripts.python.Commands",
    "runner": "PROCESS",
    "beforeCommands": ["pip install pandas -q"],
    "script": (
        "import pandas as pd\n"
        "print('Running on worker process')\n"
        "df = pd.DataFrame({'x': range(10)})\n"
        "Kestra.outputs({'sum': int(df['x'].sum())})\n"
    )
}

# DOCKER runner
docker_task = {
    "id": "isolated_task",
    "type": "io.kestra.plugin.scripts.python.Commands",
    "runner": "DOCKER",
    "docker": {
        "image": "python:3.11-slim",
        "pullPolicy": "IF_NOT_PRESENT"  # don't re-pull if already cached
    },
    "beforeCommands": [
        "pip install pandas scikit-learn -q"
    ],
    "script": (
        "import pandas as pd\n"
        "from sklearn.preprocessing import StandardScaler\n"
        "print('Running in Docker container')\n"
        "df = pd.DataFrame({'a': [1.0, 2.0, 3.0, 4.0, 5.0]})\n"
        "scaler = StandardScaler()\n"
        "scaled = scaler.fit_transform(df)\n"
        "Kestra.outputs({'mean': float(df['a'].mean()), 'std': float(df['a'].std())})\n"
    )
}

comparison_flow = {
    "id": "runner-comparison",
    "namespace": "tutorial.day09",
    "tasks": [process_task, docker_task]
}

print(yaml.dump(comparison_flow, default_flow_style=False, sort_keys=False))

## 4. inputFiles — Passing Helper Modules

`inputFiles` injects files into the task's working directory before the script runs. You can provide:
- **Inline content** — YAML string that becomes file content
- **Internal storage URI** — from a previous task's `outputFiles`

This lets you share utility modules, config files, and data between tasks without a shared filesystem.

In [ ]:
# Helper module content (inline in YAML)
helper_module = (
    "import pandas as pd\n"
    "\n"
    "def load_and_clean(url):\n"
    "    df = pd.read_csv(url)\n"
    "    df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]\n"
    "    df = df.dropna()\n"
    "    return df\n"
    "\n"
    "def compute_stats(df):\n"
    "    numeric = df.select_dtypes(include='number')\n"
    "    return {\n"
    "        'row_count': len(df),\n"
    "        'null_count': int(df.isnull().sum().sum()),\n"
    "        'numeric_cols': len(numeric.columns),\n"
    "        'total_sum': float(numeric.values.sum())\n"
    "    }\n"
)

multi_file_task = {
    "id": "use_helper",
    "type": "io.kestra.plugin.scripts.python.Commands",
    "beforeCommands": ["pip install pandas -q"],
    "inputFiles": {
        "utils.py": helper_module
    },
    "script": (
        "from utils import load_and_clean, compute_stats\n"
        "url = 'https://people.sc.fsu.edu/~jburkardt/data/csv/airtravel.csv'\n"
        "df = load_and_clean(url)\n"
        "stats = compute_stats(df)\n"
        "print(stats)\n"
        "Kestra.outputs(stats)\n"
    )
}

flow_with_helpers = {
    "id": "multi-file-task",
    "namespace": "tutorial.day09",
    "tasks": [multi_file_task]
}

print(yaml.dump(flow_with_helpers, default_flow_style=False, sort_keys=False))

In [ ]:
# Simulate the helper module execution locally
import pandas as pd

def load_and_clean(url):
    df = pd.read_csv(url)
    df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
    df = df.dropna()
    return df

def compute_stats(df):
    numeric = df.select_dtypes(include='number')
    return {
        'row_count': len(df),
        'null_count': int(df.isnull().sum().sum()),
        'numeric_cols': len(numeric.columns),
        'total_sum': float(numeric.values.sum())
    }

url = 'https://people.sc.fsu.edu/~jburkardt/data/csv/airtravel.csv'
df = load_and_clean(url)
stats = compute_stats(df)
print("Task output (Kestra.outputs payload):")
import json
print(json.dumps(stats, indent=2))

## 5. Chaining Two Python Tasks

The output of a Python task flows to the next via Pebble. The pattern:
1. Task A: compute → `Kestra.outputs({'result': value})`
2. Task B: receive result via `env: {RESULT: '{{ outputs.task_a.vars.result }}'}` or embedded in `script`

In [ ]:
# Two-stage Python pipeline
chained_flow = {
    "id": "chained-python",
    "namespace": "tutorial.day09",
    "description": "Task A aggregates data; Task B uses the output to generate a report",
    "tasks": [
        {
            "id": "aggregate",
            "type": "io.kestra.plugin.scripts.python.Commands",
            "beforeCommands": ["pip install pandas -q"],
            "script": (
                "import pandas as pd, json\n"
                "url = 'https://people.sc.fsu.edu/~jburkardt/data/csv/airtravel.csv'\n"
                "df = pd.read_csv(url)\n"
                "df.columns = [c.strip() for c in df.columns]\n"
                "# Melt wide → long for year-over-year comparison\n"
                "df_long = df.melt(id_vars=['Month'], var_name='Year', value_name='Passengers')\n"
                "by_year = df_long.groupby('Year')['Passengers'].agg(['sum', 'mean', 'max']).reset_index()\n"
                "by_year.columns = ['year', 'total', 'avg', 'peak']\n"
                "summary = by_year.to_dict('records')\n"
                "Kestra.outputs({'summary': summary, 'years': list(by_year['year'])})\n"
            )
        },
        {
            "id": "format_report",
            "type": "io.kestra.plugin.scripts.python.Commands",
            "beforeCommands": [],
            "env": {
                "SUMMARY": "{{ outputs.aggregate.vars.summary }}",
                "YEARS": "{{ outputs.aggregate.vars.years }}"
            },
            "script": (
                "import json, os\n"
                "summary = json.loads(os.environ['SUMMARY'].replace(\"'\", '\"'))\n"
                "years = json.loads(os.environ['YEARS'].replace(\"'\", '\"'))\n"
                "print(f'Report covers {len(years)} years: {years}')\n"
                "for row in summary:\n"
                "    print(f\"  {row['year']}: total={row['total']:,.0f} avg={row['avg']:.1f} peak={row['peak']}\")\n"
                "Kestra.outputs({'report_lines': len(summary)})\n"
            )
        },
        {
            "id": "log_complete",
            "type": "io.kestra.plugin.core.log.Log",
            "message": "Report generated — {{ outputs.format_report.vars.report_lines }} lines"
        }
    ]
}

print(yaml.dump(chained_flow, default_flow_style=False, sort_keys=False))

In [ ]:
# Simulate the chained pipeline locally
import pandas as pd

# Task A: aggregate
url = 'https://people.sc.fsu.edu/~jburkardt/data/csv/airtravel.csv'
df = pd.read_csv(url)
df.columns = [c.strip() for c in df.columns]
df_long = df.melt(id_vars=['Month'], var_name='Year', value_name='Passengers')
by_year = df_long.groupby('Year')['Passengers'].agg(['sum', 'mean', 'max']).reset_index()
by_year.columns = ['year', 'total', 'avg', 'peak']
summary = by_year.to_dict('records')
print("Task A — aggregate output:")
print(f"  years: {list(by_year['year'])}")

# Task B: format_report
print("\nTask B — format_report output:")
for row in summary:
    print(f"  {row['year']}: total={row['total']:,.0f} avg={row['avg']:.1f} peak={row['peak']}")

## 6. Writing Output Files from Python

In [ ]:
# Python task that writes a CSV and captures it as an output file
file_output_flow = {
    "id": "python-file-output",
    "namespace": "tutorial.day09",
    "tasks": [
        {
            "id": "generate_report",
            "type": "io.kestra.plugin.scripts.python.Commands",
            "beforeCommands": ["pip install pandas -q"],
            "script": (
                "import pandas as pd\n"
                "url = 'https://people.sc.fsu.edu/~jburkardt/data/csv/airtravel.csv'\n"
                "df = pd.read_csv(url)\n"
                "df.columns = [c.strip() for c in df.columns]\n"
                "# Write summary CSV\n"
                "numeric_df = df.select_dtypes(include='number')\n"
                "summary = numeric_df.describe().T\n"
                "summary.to_csv('summary.csv')\n"
                "print(f'Written {len(summary)} stat rows to summary.csv')\n"
                "Kestra.outputs({'rows_written': len(summary)})\n"
            ),
            "outputFiles": ["*.csv"]
        },
        {
            "id": "read_report",
            "type": "io.kestra.plugin.scripts.python.Commands",
            "beforeCommands": ["pip install pandas -q"],
            "inputFiles": {
                "data.csv": "{{ outputs.generate_report.outputFiles['summary.csv'] }}"
            },
            "script": (
                "import pandas as pd\n"
                "df = pd.read_csv('data.csv', index_col=0)\n"
                "print(df.to_string())\n"
                "Kestra.outputs({'columns_in_report': len(df)})\n"
            )
        }
    ]
}

print(yaml.dump(file_output_flow, default_flow_style=False, sort_keys=False))

## 7. Environment Variables — Secure Parameter Passing

In [ ]:
# Pattern: pass upstream outputs and secrets to Python via env vars
secure_task = {
    "id": "secure_api_call",
    "type": "io.kestra.plugin.scripts.python.Commands",
    "beforeCommands": ["pip install requests -q"],
    "env": {
        # Secrets masked in logs
        "API_KEY": "{{ secret('MY_API_KEY') }}",
        # Values from upstream task outputs
        "ROW_COUNT": "{{ outputs.aggregate.vars.row_count }}",
        # Flow inputs
        "TARGET_ENV": "{{ inputs.environment }}"
    },
    "script": (
        "import os, requests\n"
        "api_key = os.environ['API_KEY']  # masked in Kestra logs\n"
        "row_count = int(os.environ.get('ROW_COUNT', '0'))\n"
        "env = os.environ.get('TARGET_ENV', 'dev')\n"
        "print(f'Posting {row_count} rows to {env} API (key: {api_key[:4]}***)')\n"
        "# Make the API call using requests...\n"
        "Kestra.outputs({'posted': row_count, 'env': env})\n"
    )
}

print("env: block pattern:")
print(yaml.dump({"env": secure_task["env"]}, default_flow_style=False, sort_keys=False))

# Simulate locally
import os
os.environ['API_KEY'] = 'sk-test-1234'
os.environ['ROW_COUNT'] = '144'
os.environ['TARGET_ENV'] = 'staging'

api_key = os.environ['API_KEY']
row_count = int(os.environ['ROW_COUNT'])
env = os.environ['TARGET_ENV']
print(f"\nSimulation: Posting {row_count} rows to {env} (key: {api_key[:4]}***)")

## 8. Full Python-Centric Pipeline

In [ ]:
full_python_flow = {
    "id": "full-python-pipeline",
    "namespace": "tutorial.day09",
    "description": "ingest → transform → validate → export",
    "inputs": [
        {"id": "source_url", "type": "STRING",
         "defaults": "https://people.sc.fsu.edu/~jburkardt/data/csv/airtravel.csv"},
        {"id": "environment", "type": "STRING", "defaults": "dev"}
    ],
    "tasks": [
        {
            "id": "ingest",
            "type": "io.kestra.plugin.scripts.python.Commands",
            "beforeCommands": ["pip install pandas -q"],
            "env": {"SOURCE_URL": "{{ inputs.source_url }}"},
            "script": (
                "import pandas as pd, os\n"
                "df = pd.read_csv(os.environ['SOURCE_URL'])\n"
                "df.columns = [c.strip() for c in df.columns]\n"
                "df.to_csv('raw.csv', index=False)\n"
                "Kestra.outputs({'rows': len(df), 'cols': len(df.columns)})\n"
            ),
            "outputFiles": ["raw.csv"],
            "retry": {"type": "constant", "maxAttempts": 3, "delay": "PT5S"}
        },
        {
            "id": "transform",
            "type": "io.kestra.plugin.scripts.python.Commands",
            "beforeCommands": ["pip install pandas -q"],
            "inputFiles": {"raw.csv": "{{ outputs.ingest.outputFiles['raw.csv'] }}"},
            "script": (
                "import pandas as pd\n"
                "df = pd.read_csv('raw.csv')\n"
                "numeric_df = df.select_dtypes(include='number')\n"
                "summary = numeric_df.describe().round(2)\n"
                "summary.to_csv('summary.csv')\n"
                "Kestra.outputs({'summary_rows': len(summary)})\n"
            ),
            "outputFiles": ["summary.csv"]
        },
        {
            "id": "log_complete",
            "type": "io.kestra.plugin.core.log.Log",
            "message": (
                "[{{ inputs.environment | upper }}] Pipeline done — "
                "{{ outputs.ingest.vars.rows }} rows ingested, "
                "{{ outputs.transform.vars.summary_rows }} summary rows written"
            )
        }
    ]
}

print(yaml.dump(full_python_flow, default_flow_style=False, sort_keys=False))

## Challenge

Build a two-task Python pipeline:

**Task A (`fetch_and_pivot`):**
- Read `https://people.sc.fsu.edu/~jburkardt/data/csv/airtravel.csv` with pandas
- Melt wide → long format (Month as id_var)
- Compute total passengers per month across all years
- Call `Kestra.outputs({'top_month': str_name, 'top_count': int_count})`
- Write the long-format DataFrame to `monthly.csv` and capture with `outputFiles`

**Task B (`format_insight`):**
- Receive `top_month` and `top_count` from Task A via `env:`
- Read `monthly.csv` from `inputFiles`
- Print: `"Peak month: {top_month} with {top_count:,} passengers across all years"`
- Call `Kestra.outputs({'insight': the_string})`

**Task C:** Log `"Insight: {{ outputs.format_insight.vars.insight }}"`

In [ ]:
# Your solution here
challenge_flow = {
    "id": "airtravel-insights",
    "namespace": "tutorial.day09.challenge",
    "tasks": [
        # Task A: fetch_and_pivot
        # Task B: format_insight
        # Task C: log
    ]
}
print(yaml.dump(challenge_flow, default_flow_style=False, sort_keys=False))

## Recap

| Pattern | YAML construct | Notes |
|---------|---------------|-------|
| Run Python | `type: scripts.python.Commands` | Inject deps with `beforeCommands` |
| Publish output | `Kestra.outputs({'k': v})` | No import; JSON-serialisable values only |
| Reference output | `{{ outputs.id.vars.k }}` | Available in any Pebble template downstream |
| Pass file in | `inputFiles: {name: URI}` | URI from previous task's `outputFiles` |
| Pass file out | `outputFiles: ['*.csv']` | Files go to Kestra internal storage |
| Secure values | `env: {KEY: '{{ secret("K") }}'}` | Masked in UI and logs |
| Docker isolation | `runner: DOCKER` + `docker.image` | Full env control, cached after first run |

**Tip:** Use `Kestra.outputs({'key': value})` at the end of your Python script to expose computed values to downstream tasks. The Kestra runtime injects this function automatically — no import needed.

**Tomorrow — Day 10:** Capstone — build a complete end-to-end data pipeline with Kestra.